In [1]:
import pygame
import random

# Initialize pygame
pygame.init()

# Game Constants
WIDTH, HEIGHT = 600, 400
BLOCK_SIZE = 25  # Increased snake size
SNAKE_SPEEDS = {'Easy': 10, 'Medium': 15, 'Hard': 20}

# Colors
SNAKE_SKINS = {
    "Black": (0, 0, 0),
    "Red": (200, 0, 0),
    "Blue": (0, 0, 200),
    "Green": (0, 150, 0)
}
EYE_COLOR = (255, 255, 255)  # White eyes
FOOD_COLORS = [(255, 102, 102), (102, 255, 102), (102, 102, 255), (255, 255, 102)]
TEXT_COLOR = (0, 0, 0)
BUTTON_COLOR = (255, 215, 0)
BUTTON_HIGHLIGHT = (255, 165, 0)
BG_GRADIENT_TOP = (200, 230, 255)
BG_GRADIENT_BOTTOM = (240, 255, 255)
BORDER_COLOR = (0, 0, 0)  # Border for the game area

# Initialize game window
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Snake Game")

# Clock for controlling speed
clock = pygame.time.Clock()

# Font for displaying text
font = pygame.font.SysFont("bahnschrift", 25)

def draw_background():
    for y in range(HEIGHT):
        color = [
            BG_GRADIENT_TOP[i] + (BG_GRADIENT_BOTTOM[i] - BG_GRADIENT_TOP[i]) * y // HEIGHT 
            for i in range(3)
        ]
        pygame.draw.line(screen, color, (0, y), (WIDTH, y))
    pygame.draw.rect(screen, BORDER_COLOR, (0, 0, WIDTH, HEIGHT), 5)

def draw_score(score):
    score_text = font.render(f"Score: {score}", True, TEXT_COLOR)
    screen.blit(score_text, (WIDTH - 120, 10))

def draw_high_score(high_score):
    high_score_text = font.render(f"High Score: {high_score}", True, TEXT_COLOR)
    screen.blit(high_score_text, (WIDTH - 180, 40))

def draw_name():
    name_text = font.render("Aryan's Snake Game", True, TEXT_COLOR)
    screen.blit(name_text, (WIDTH // 2 - name_text.get_width() // 2, 10))

def draw_buttons(button_texts, title):
    screen.fill((220, 220, 220))
    title_text = font.render(title, True, TEXT_COLOR)
    screen.blit(title_text, (WIDTH // 2 - 80, HEIGHT // 4 - 30))
    
    buttons = []
    for i, text in enumerate(button_texts):
        rect = pygame.Rect(WIDTH // 3, HEIGHT // 3 + i * 50, 200, 40)
        buttons.append((rect, text))
        pygame.draw.rect(screen, BUTTON_COLOR, rect, border_radius=10)
        label = font.render(text, True, TEXT_COLOR)
        screen.blit(label, (rect.x + 60, rect.y + 10))
    
    pygame.display.update()
    return buttons

def select_option(options, title):
    buttons = draw_buttons(options, title)
    while True:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                quit()
            elif event.type == pygame.MOUSEBUTTONDOWN:
                for rect, option in buttons:
                    if rect.collidepoint(event.pos):
                        return option

def game_over_screen(high_score):
    buttons = draw_buttons(["Replay", "Quit"], "Game Over")
    draw_high_score(high_score)
    pygame.display.update()
    while True:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                quit()
            elif event.type == pygame.MOUSEBUTTONDOWN:
                if buttons[0][0].collidepoint(event.pos):
                    game_loop()
                elif buttons[1][0].collidepoint(event.pos):
                    pygame.quit()
                    quit()

def game_loop():
    SNAKE_SPEED = SNAKE_SPEEDS[select_option(["Easy", "Medium", "Hard"], "Select Difficulty")]
    SNAKE_COLOR = SNAKE_SKINS[select_option(["Black", "Red", "Blue", "Green"], "Select Snake Skin")]
    
    game_over = False
    score = 0
    high_score = 0
    x, y = WIDTH // 2, HEIGHT // 2
    dx, dy = 0, 0
    snake_body = []
    snake_length = 1

    food_x = round(random.randrange(0, WIDTH - BLOCK_SIZE) / BLOCK_SIZE) * BLOCK_SIZE
    food_y = round(random.randrange(0, HEIGHT - BLOCK_SIZE) / BLOCK_SIZE) * BLOCK_SIZE
    food_color = random.choice(FOOD_COLORS)

    while not game_over:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                game_over = True
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_LEFT and dx == 0:
                    dx, dy = -BLOCK_SIZE, 0
                elif event.key == pygame.K_RIGHT and dx == 0:
                    dx, dy = BLOCK_SIZE, 0
                elif event.key == pygame.K_UP and dy == 0:
                    dx, dy = 0, -BLOCK_SIZE
                elif event.key == pygame.K_DOWN and dy == 0:
                    dx, dy = 0, BLOCK_SIZE
        
        x += dx
        y += dy
        
        if x >= WIDTH or x < 0 or y >= HEIGHT or y < 0:
            game_over = True
        
        snake_body.append([x, y])
        if len(snake_body) > snake_length:
            del snake_body[0]
        
        for block in snake_body[:-1]:
            if block == [x, y]:
                game_over = True
        
        draw_background()
        draw_name()
        draw_score(score)
        draw_high_score(high_score)
        pygame.draw.circle(screen, food_color, (food_x + BLOCK_SIZE//2, food_y + BLOCK_SIZE//2), BLOCK_SIZE//2)
        for block in snake_body:
            pygame.draw.rect(screen, SNAKE_COLOR, [block[0], block[1], BLOCK_SIZE, BLOCK_SIZE])
        pygame.draw.circle(screen, EYE_COLOR, (x + BLOCK_SIZE//3, y + BLOCK_SIZE//3), BLOCK_SIZE//6)
        pygame.draw.circle(screen, EYE_COLOR, (x + 2*BLOCK_SIZE//3, y + BLOCK_SIZE//3), BLOCK_SIZE//6)
        pygame.display.update()
        
        if x == food_x and y == food_y:
            food_x = round(random.randrange(0, WIDTH - BLOCK_SIZE) / BLOCK_SIZE) * BLOCK_SIZE
            food_y = round(random.randrange(0, HEIGHT - BLOCK_SIZE) / BLOCK_SIZE) * BLOCK_SIZE
            food_color = random.choice(FOOD_COLORS)
            snake_length += 1
            score += 10
            high_score = max(high_score, score)
        
        clock.tick(SNAKE_SPEED)
    
    game_over_screen(high_score)

game_loop()



pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


error: video system not initialized